# Riesgo de crédito de emisores de energía en Colombia — 01: extracción de datos XBRL

Lee los 55 archivos XBRL de la Superintendencia Financiera (SIMEV / RNVE), 5 emisores x 11 años (2015–2025), y construye
una tabla con 17 cifras por emisor y año en **billones de pesos**.

- **Entrada**: carpeta `Prueba_Bancolombia/<EMISOR>/*.xbrl` en Drive (ISA, ISAGEN, EPM, CELSIA, ENEL).
- **Salida**: `tabla_emisores.csv` (55 filas x 17 cifras), que usa el cuaderno 02.
- **Por qué un lector propio**: Arelle (la librería que sugiere la prueba) necesita descargar la taxonomía del servidor de la SFC,
  que respondía "Bad Gateway". Los números y los nombres IFRS ya vienen dentro del archivo, así que se lee directamente como XML con `lxml`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
from lxml import etree

RUTA = "/content/drive/MyDrive/Prueba_Bancolombia"
print({e: len(os.listdir(f"{RUTA}/{e}")) for e in ["ISA", "ISAGEN", "EPM", "CELSIA", "ENEL"]})

## 1. Lector XBRL
Un XBRL es un XML donde cada cifra viene con un nombre estándar (`ifrs:Revenue`, `ifrs:Assets`...) y un **contexto** que dice a qué
fecha o periodo pertenece. El lector arma el diccionario de contextos y devuelve todos los hechos numéricos del archivo.

In [ ]:
def leer_xbrl(ruta):
    """Lee un archivo XBRL de la Superfinanciera y devuelve una tabla con todas las cifras."""
    raiz = etree.parse(ruta).getroot()
    XBRLI = "{http://www.xbrl.org/2003/instance}"
    XBRLDI = "{http://xbrl.org/2006/xbrldi}"

    # 1. Contextos: a qué fecha o periodo pertenece cada cifra, y si es un total o un desglose (con dimensión)
    contextos = {}
    for c in raiz.findall(XBRLI + "context"):
        periodo = c.find(XBRLI + "period")
        instante = periodo.find(XBRLI + "instant")
        inicio = periodo.find(XBRLI + "startDate")
        fin = periodo.find(XBRLI + "endDate")
        con_dimension = (c.find(".//" + XBRLDI + "explicitMember") is not None
                         or c.find(".//" + XBRLDI + "typedMember") is not None)
        contextos[c.get("id")] = {
            "fecha": instante.text if instante is not None else fin.text,
            "inicio": inicio.text if inicio is not None else None,
            "con_dimension": con_dimension,
        }

    # 2. Hechos: cada cifra reportada (solo las numéricas, que son las que tienen unidad)
    filas = []
    for el in raiz.iter():
        if not isinstance(el.tag, str) or el.get("contextRef") is None or el.get("unitRef") is None:
            continue
        q = etree.QName(el)
        prefijo = next((p for p, u in raiz.nsmap.items() if u == q.namespace), "?")
        ctx = contextos[el.get("contextRef")]
        filas.append({
            "cuenta": f"{prefijo}:{q.localname}",
            "valor": pd.to_numeric(el.text, errors="coerce"),
            "contexto": el.get("contextRef"),
            "fecha": ctx["fecha"],
            "inicio": ctx["inicio"],
            "con_dimension": ctx["con_dimension"],
            "unidad": el.get("unitRef"),
        })
    return pd.DataFrame(filas)

## 2. Las 11 cifras base
Reglas de extracción (cada una existe porque un emisor la hizo necesaria):
- Los contextos se eligen por **fechas** (flujo 1 ene – 31 dic; saldo 31 dic), nunca por nombre: cada emisor nombra sus contextos distinto.
- La **escala** se detecta sola: activo total mayor que 10¹² = el archivo está en pesos (ISAGEN); si no, en miles de pesos. Todo se convierte a billones.
- Cada cifra tiene una **lista de cuentas alternativas**; se usa la primera que exista y no sea cero.
- Si una cifra falta en el archivo de un año, se toma del **comparativo** que trae el archivo del año siguiente.
- **Control de coherencia**: si ingresos < 0,9 x (utilidad bruta + costo de ventas), se usa la suma (Celsia 2025 etiquetó mal sus ingresos).
- La depreciación se busca también en las **notas** cuando no está en la línea principal (Enel). Enel 2015 se toma del PDF (nota 24: 0,16 billones).

In [ ]:
# Cada cifra tiene una lista de alternativas. Una alternativa puede ser una cuenta o una tupla de cuentas que se suman.
CUENTAS = {
    "ingresos": ["ifrs:Revenue"],
    "utilidad_operacional": ["ifrs:ProfitLossFromOperatingActivities"],
    "dya": ["ifrs:AdjustmentsForDepreciationAndAmortisationExpense", "ifrs:DepreciationAndAmortisationExpense"],
    "intereses": ["ifrs:FinanceCosts", "ifrs:InterestExpense"],
    "utilidad_neta": ["ifrs:ProfitLoss"],
    "deuda": [("co-sfc-core:ObligacionesFinancierasCorrientes",
               "co-sfc-core:ObligacionesFinancierasNoCorrientes",
               "co-sfc-core:TitulosEmitidos"),
              "ifrs:Borrowings"],
    "caja": ["ifrs:CashAndCashEquivalents"],
    "activos": ["ifrs:Assets"],
    "activo_corriente": ["ifrs:CurrentAssets"],
    "pasivo_corriente": ["ifrs:CurrentLiabilities"],
    "patrimonio": ["ifrs:Equity"],
}

def valor(h, cuenta):
    """Valor de una cuenta (o suma de una tupla de cuentas). None si no existe o es cero."""
    if isinstance(cuenta, tuple):
        partes = [valor(h, c) for c in cuenta]
        partes = [p for p in partes if p is not None]
        return sum(partes) if partes else None
    v = h.loc[h.cuenta == cuenta, "valor"].dropna()
    v = v[v != 0]
    return v.iloc[0] if len(v) else None

def maximo(h, cuenta):
    """Mayor valor de una cuenta en cualquier contexto del periodo (el total de una nota con desglose)."""
    v = h.loc[h.cuenta == cuenta, "valor"].dropna()
    v = v[v > 0]
    return v.max() if len(v) else None

def cifras_del_anio(h, emisor, anio, escala, origen):
    """Las 11 cifras de un año dentro de un archivo (sirve para el año propio y el comparativo)."""
    hh = h[~h.con_dimension & (h.fecha == f"{anio}-12-31") & (h.inicio.isna() | (h.inicio == f"{anio}-01-01"))]
    fila = {"emisor": emisor, "anio": anio, "origen": origen}
    for nombre, alternativas in CUENTAS.items():
        fila[nombre] = None
        for alt in alternativas:
            v = valor(hh, alt)
            if v is not None:
                fila[nombre] = round(v / escala, 2)     # -> billones de COP
                break
    # Control de coherencia: los ingresos no pueden ser menores que utilidad bruta + costo de ventas
    bruto = valor(hh, ("ifrs:GrossProfit", "ifrs:CostOfSales"))
    if bruto is not None and fila["ingresos"] is not None and fila["ingresos"] < 0.9 * bruto / escala:
        fila["ingresos"] = round(bruto / escala, 2)
    # Respaldo para D&A: algunos emisores (Enel) solo la reportan en las notas (con desglose)
    if fila["dya"] is None:
        hd = h[(h.fecha == f"{anio}-12-31") & (h.inicio == f"{anio}-01-01")]
        total = maximo(hd, "ifrs:DepreciationAndAmortisationExpense")
        if total is None:
            partes = [maximo(hd, "ifrs:DepreciationPropertyPlantAndEquipment"),
                      maximo(hd, "ifrs:AmortisationIntangibleAssetsOtherThanGoodwill")]
            partes = [p for p in partes if p is not None]
            total = sum(partes) if partes else None
        if total is not None:
            fila["dya"] = round(total / escala, 2)
    return fila

def extraer_archivo(archivo, emisor):
    anio = int(archivo[-15:-11])
    h = leer_xbrl(archivo)
    activos = h.loc[~h.con_dimension & (h.cuenta == "ifrs:Assets"), "valor"].max()
    escala = 1e12 if activos > 1e12 else 1e9          # pesos vs miles de pesos
    return [cifras_del_anio(h, emisor, anio, escala, "propio"),
            cifras_del_anio(h, emisor, anio - 1, escala, "comparativo")]

def construir_tabla(RUTA):
    filas = []
    for emisor in sorted(os.listdir(RUTA)):
        if os.path.isdir(f"{RUTA}/{emisor}"):
            for a in sorted(os.listdir(f"{RUTA}/{emisor}")):
                if a.endswith(".xbrl"):
                    filas += extraer_archivo(f"{RUTA}/{emisor}/{a}", emisor)
    df = pd.DataFrame(filas)
    # Para cada emisor-año: el valor del archivo propio; si falta, el comparativo del archivo siguiente
    df["orden"] = (df.origen != "propio").astype(int)
    df = df.sort_values(["emisor", "anio", "orden"])
    tabla = df.groupby(["emisor", "anio"], as_index=False).first().drop(columns=["orden", "origen"])
    return tabla[tabla.anio >= 2015].reset_index(drop=True)

tabla = construir_tabla(RUTA)
tabla.loc[(tabla.emisor == "ENEL") & (tabla.anio == 2015), "dya"] = 0.16   # PDF EEFF Emgesa 2015, p. 8, nota 24
tabla.to_csv(f"{RUTA}/tabla_emisores.csv", index=False)
print("Filas:", len(tabla), "| Vacíos:", int(tabla.isna().sum().sum()))
tabla[tabla.anio == 2025]

## 3. Las 6 cifras del flujo de caja (prueba de liquidez de S&P)
Fuentes de caja = caja + FFO (EBITDA − intereses pagados − impuestos pagados). Usos = deuda que vence en 12 meses + capex + dividendos.
Para las salidas de caja se toma la **mayor** de las líneas alternativas, porque algunos archivos etiquetan solo una parte.
ISAGEN 2016, 2018 y 2020: el XBRL no registra ninguna línea de dividendos (en financiación solo hay movimientos de deuda); se dejan en cero.

In [ ]:
CUENTAS_LIQ = {
    "flujo_operativo":   ["ifrs:CashFlowsFromUsedInOperatingActivities"],
    "deuda_corto_plazo": ["co-sfc-core:ObligacionesFinancierasCorrientes",
                          "ifrs:CurrentBorrowingsAndCurrentPortionOfNoncurrentBorrowings", "ifrs:ShorttermBorrowings"],
    "intereses_pagados": [("ifrs:InterestPaidClassifiedAsOperatingActivities", "ifrs:InterestPaidClassifiedAsFinancingActivities"),
                          "ifrs:FinanceCostsPaidClassifiedAsOperatingActivities"],
    "impuestos_pagados": ["ifrs:IncomeTaxesPaidRefundClassifiedAsOperatingActivities",
                          "ifrs:IncomeTaxesPaidClassifiedAsOperatingActivities", "ifrs:IncomeTaxesPaidRefund"],
    "capex":             ["ifrs:PurchaseOfPropertyPlantAndEquipmentIntangibleAssetsOtherThanGoodwillInvestmentPropertyAndOtherNoncurrentAssets",
                          ("ifrs:PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
                           "ifrs:PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities")],
    "dividendos_pagados": ["ifrs:DividendsPaidClassifiedAsFinancingActivities",
                           ("ifrs:DividendsPaidToEquityHoldersOfParentClassifiedAsFinancingActivities",
                            "ifrs:DividendsPaidToNoncontrollingInterestsClassifiedAsFinancingActivities"),
                           "ifrs:DividendsPaidOrdinaryShares"],
}
PRIMERO = {"flujo_operativo", "deuda_corto_plazo"}      # se usa la primera alternativa que exista
SALIDAS = {"intereses_pagados", "impuestos_pagados", "capex", "dividendos_pagados"}   # se usa la mayor, en valor absoluto

def cifras_liquidez(h, emisor, anio, escala, origen):
    hh = h[~h.con_dimension & (h.fecha == f"{anio}-12-31") & (h.inicio.isna() | (h.inicio == f"{anio}-01-01"))]
    fila = {"emisor": emisor, "anio": anio, "origen": origen}
    for nombre, alternativas in CUENTAS_LIQ.items():
        vals = [valor(hh, alt) for alt in alternativas]
        vals = [abs(v) if nombre in SALIDAS else v for v in vals if v is not None]
        if not vals:
            fila[nombre] = None
        elif nombre in PRIMERO:
            fila[nombre] = round(vals[0] / escala, 2)
        else:
            fila[nombre] = round(max(vals) / escala, 2)
    return fila

def extraer_liquidez(archivo, emisor):
    anio = int(archivo[-15:-11])
    h = leer_xbrl(archivo)
    activos = h.loc[~h.con_dimension & (h.cuenta == "ifrs:Assets"), "valor"].max()
    escala = 1e12 if activos > 1e12 else 1e9
    return [cifras_liquidez(h, emisor, anio, escala, "propio"),
            cifras_liquidez(h, emisor, anio - 1, escala, "comparativo")]

filas = []
for emisor in sorted(os.listdir(RUTA)):
    if os.path.isdir(f"{RUTA}/{emisor}"):
        for a in sorted(os.listdir(f"{RUTA}/{emisor}")):
            if a.endswith(".xbrl"):
                filas += extraer_liquidez(f"{RUTA}/{emisor}/{a}", emisor)
liq = pd.DataFrame(filas)
liq["orden"] = (liq.origen != "propio").astype(int)
liq = liq.sort_values(["emisor", "anio", "orden"]).groupby(["emisor", "anio"], as_index=False).first().drop(columns=["orden", "origen"])
liq = liq[liq.anio >= 2015]

tabla = tabla.drop(columns=[c for c in CUENTAS_LIQ if c in tabla.columns]).merge(liq, on=["emisor", "anio"], how="left")
tabla["dividendos_pagados"] = tabla["dividendos_pagados"].fillna(0)   # ISAGEN 2016, 2018, 2020: sin línea de dividendos en el XBRL
tabla.to_csv(f"{RUTA}/tabla_emisores.csv", index=False)
print("Filas:", len(tabla), "| Vacíos:", int(tabla.isna().sum().sum()))
tabla[tabla.anio == 2025][["emisor", "caja", "flujo_operativo", "intereses_pagados", "impuestos_pagados", "deuda_corto_plazo", "capex", "dividendos_pagados"]]

## 4. Las 4 cifras del modelo de Altman
El modelo Z de Altman no se alimenta de los mismos rubros que el scorecard de S&P: necesita utilidades retenidas, plusvalía e
intangibles (para llegar a **activos tangibles**, que es el denominador que usa Bloomberg) y el pasivo total.
- `ifrs:RetainedEarnings` — utilidades retenidas acumuladas.
- `ifrs:Goodwill` — plusvalía; no todos los emisores la reportan, se rellena con 0.
- `ifrs:IntangibleAssetsOtherThanGoodwill` (o `ifrs:IntangibleAssetsAndGoodwill`) — intangibles.
- `ifrs:Liabilities` — pasivo total; si falta, se calcula como activo total menos patrimonio.

Se guardan con **4 decimales**, no con 2: son partidas chicas y redondear a dos las destruye.

In [ ]:
# Las 4 cifras que necesita el modelo de Altman. Se guardan con 4 decimales porque
# son partidas pequeñas (las utilidades retenidas de Celsia son -0,15 billones).
CUENTAS_ALTMAN = {
    "utilidades_retenidas": ["ifrs:RetainedEarnings"],
    "goodwill":             ["ifrs:Goodwill"],
    "intangibles":          ["ifrs:IntangibleAssetsOtherThanGoodwill", "ifrs:IntangibleAssetsAndGoodwill"],
    "pasivo_total":         ["ifrs:Liabilities"],
}

def cifras_altman(h, emisor, anio, escala, origen):
    hh = h[~h.con_dimension & (h.fecha == f"{anio}-12-31") & (h.inicio.isna() | (h.inicio == f"{anio}-01-01"))]
    fila = {"emisor": emisor, "anio": anio, "origen": origen}
    for nombre, alternativas in CUENTAS_ALTMAN.items():
        fila[nombre] = None
        for alt in alternativas:
            v = valor(hh, alt)
            if v is not None:
                fila[nombre] = round(v / escala, 4)
                break
    return fila

def extraer_altman(archivo, emisor):
    anio = int(archivo[-15:-11])
    h = leer_xbrl(archivo)
    activos = h.loc[~h.con_dimension & (h.cuenta == "ifrs:Assets"), "valor"].max()
    escala = 1e12 if activos > 1e12 else 1e9
    return [cifras_altman(h, emisor, anio, escala, "propio"),
            cifras_altman(h, emisor, anio - 1, escala, "comparativo")]

filas = []
for emisor in sorted(os.listdir(RUTA)):
    if os.path.isdir(f"{RUTA}/{emisor}"):
        for a in sorted(os.listdir(f"{RUTA}/{emisor}")):
            if a.endswith(".xbrl"):
                filas += extraer_altman(f"{RUTA}/{emisor}/{a}", emisor)
alt = pd.DataFrame(filas)
alt["orden"] = (alt.origen != "propio").astype(int)
alt = alt.sort_values(["emisor", "anio", "orden"]).groupby(["emisor", "anio"], as_index=False).first().drop(columns=["orden", "origen"])
alt = alt[alt.anio >= 2015]

tabla = tabla.drop(columns=[c for c in CUENTAS_ALTMAN if c in tabla.columns]).merge(alt, on=["emisor", "anio"], how="left")
tabla["goodwill"] = tabla.goodwill.fillna(0)                                      # no todos reportan plusvalía
tabla["pasivo_total"] = tabla.pasivo_total.fillna(tabla.activos - tabla.patrimonio)  # respaldo: activo - patrimonio
tabla["activos_tangibles"] = tabla.activos - tabla.goodwill - tabla.intangibles
tabla.to_csv(f"{RUTA}/tabla_emisores.csv", index=False)
print("Filas:", len(tabla), "| Vacíos:", int(tabla.isna().sum().sum()))
tabla[tabla.anio == 2025][["emisor", "utilidades_retenidas", "goodwill", "intangibles", "activos_tangibles", "pasivo_total", "patrimonio"]]
